# Client experiments

Explore typed selections, generated fields and evaluated paths. Full records and queries are saved beside this executed notebook. The printed observations contain scope, counts and query references.


In [ ]:
import json, logging, os
from pathlib import Path
from rdfsolve.api import Client

root = Path(os.environ.get("RDFSOLVE_ROOT", "../..")).resolve()
schemas = root / "notebooks/mcp/schemas"
out = Path(os.environ.get("RDFSOLVE_OUTPUT", root.parent / "logs/mcp-test/client-direct"))
out.mkdir(parents=True, exist_ok=True)
logging.basicConfig(filename=out / "client.log", level=logging.INFO, force=True)


In [ ]:
with Client.open(schemas / "aopwikirdf.schema.json", timeout=30) as aopwiki:
    selected = aopwiki.find("lung", kind="Adverse Outcome Pathway")
    display(selected.summary())
    display(aopwiki.describe(owners=["Adverse Outcome Pathway"], targets=["Stressor"]))
    paths = selected.paths_between("Chemical entity", max_hops=2, max_paths=100, allow_partial=True)
    display(paths.attrs["observations"])
    batch = {(r["bindings"]["n0"]["value"], r["bindings"]["n2"]["value"]) for r in paths.attrs["routes"]}
    individually = set()
    for record in selected:
        one = aopwiki.paths_between(record, "Chemical entity", max_hops=2, max_paths=100)
        individually.update((r["bindings"]["n0"]["value"], r["bindings"]["n2"]["value"]) for r in one.attrs["routes"])
    assert batch == individually and batch
    print(f"{len(selected)} selected AOPs; {len(batch)} AOP/chemical associations; batch equals the individual client calls")
    aopwiki.save_session(out / "aopwiki-client.json")
    display(aopwiki.trace())


In [ ]:
for name, terms in [("aopwikirdf-small", ["decreased"]), ("wikipathways-small", ["ENSG00000100030"])]:
    with Client.open(schemas / f"{name}.schema.json", data_file=schemas / f"{name}.ttl") as data:
        matches = data.search(terms)
        assert len(matches) > 0
        display({"dataset": name, **matches.summary()})
        display(matches.paths())
        data.save_session(out / f"{name}-client.json")
        display(data.trace())
